# Alternative FC Measures — Fingerprinting Comparison (HCP 100 Unrelated)

Compare scaffold-based fingerprinting against alternative functional connectivity measures:
- **Pearson correlation** (full, th=25%, th=1%) — baseline
- **Precision matrix** (regularized inverse covariance via Ledoit-Wolf)
- **Partial correlation** (derived from precision matrix)
- **Gaussian Copula MI** (GCMI, captures nonlinear monotonic dependencies)

Reference: Liu et al. (2025) "Benchmarking methods for mapping functional connectivity
in the brain", *Nature Methods* 22:1593–1602

In [ ]:
import numpy as np
import sys
sys.path.append("..")
from topofinger import config
import pickle as pk
import os, sys
from collections import OrderedDict
from tqdm.notebook import tqdm

from sklearn.covariance import LedoitWolf
from scipy.stats import rankdata, norm

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from topofinger.fingerprinting import compute_idiff, compute_effect_size, compute_success_rate
from topofinger.hcp_IO import (load_ts_schaefer300_dict, load_scaffold_schaefer300,
                         save_correlations_subjects_fast, load_fc_schaefer300)
from topofinger.viz import plot_identifiability

In [ ]:
# === Cohort: "100" = 100 unrelated HCP benchmark; "1005" = full cohort (HCP 1200 release, 300 cortical regions) ===
COHORT = os.environ.get("ALTFC_COHORT", "100")
if COHORT == "100":
    DATA_PATH = config.HCP_100 + "/schaefer300/no_hp/"
    TS_PATH = DATA_PATH + "ts/LR/"
    SUFFIX = ""
    COHORT_LABEL = "HCP 100 Unrelated (Schaefer 300)"
else:
    DATA_PATH = config.HCP_1200 + "/NoGSR_LowPassOnly/"
    TS_PATH = DATA_PATH + "ts/"
    SUFFIX = "_cortex"
    COHORT_LABEL = "HCP 1005 (Schaefer 300, cortical)"
SCAFFOLDS_PATH = DATA_PATH + "scaffolds/whole/"
FC_PATH = DATA_PATH + "fc/whole/"
FIG_PATH = config.FIGURES_RAW + f"/alternative_fc/{COHORT}/"
os.makedirs(FIG_PATH, exist_ok=True)
print("Cohort:", COHORT_LABEL)


In [ ]:
# === Load time series ===
print("Loading time series...")
ts_data = OrderedDict()
if COHORT == "100":
    # files like 100307_REST1_LR.csv (already z-scored)
    for f in tqdm(sorted(os.listdir(TS_PATH))):
        sub = f[:6]
        sess = int(f.split('_')[1][-1]) - 1  # REST1 -> 0, REST2 -> 1
        ts_data.setdefault(sub, OrderedDict())[sess] = np.nan_to_num(np.loadtxt(TS_PATH + f, delimiter=',', skiprows=1))
else:
    # same subjects and order as the similarity matrices of scripts/06 (sorted subject IDs)
    for sub in tqdm(sorted({f[:6] for f in os.listdir(DATA_PATH + "scaffolds/whole_cortex/freq")})):
        ts_data[sub] = OrderedDict()
        for sess in (0, 1):
            f = f"{sub}_rfMRI_REST{sess + 1}_LR_Schaefer300+Tian_NoGSR_LowPassOnly.npz"
            ts = np.load(TS_PATH + f)["timeseries"][:config.N_CORTICAL].T   # (timepoints, 300)
            ts = np.nan_to_num(ts)
            ts_data[sub][sess] = ((ts - ts.mean(0)) / ts.std(0)).astype(np.float32)   # z-scored, as the benchmark files

n_sub = len(ts_data)
subs = list(ts_data.keys())
print(f"Loaded {n_sub} subjects, {len(ts_data[subs[0]])} sessions each")
print(f"TS shape per session: {ts_data[subs[0]][0].shape}")


In [ ]:
# === Load existing scaffold & FC fingerprinting results ===
print("Loading existing fingerprinting similarity matrices...")
scaff_corr = np.load(SCAFFOLDS_PATH + f"scaffold_correlations_whole{SUFFIX}.npy")
fc_corr_th100 = np.load(FC_PATH + f"fc_correlations_whole_th100{SUFFIX}.npy")
fc_corr_th25  = np.load(FC_PATH + f"fc_correlations_whole_th25{SUFFIX}.npy")
fc_corr_th1   = np.load(FC_PATH + f"fc_correlations_whole_th1{SUFFIX}.npy")
print(f"Scaffold sim: {scaff_corr.shape}, FC sim: {fc_corr_th100.shape}")
assert scaff_corr.shape[0] == n_sub, "subject count mismatch with similarity matrices"


## 1. Compute Alternative FC Measures

In [ ]:
def build_fc_dict(ts_data, compute_fn, desc="Computing"):
    """Compute per-subject per-session FC matrices using a custom function.
    compute_fn: takes (timepoints, ROIs) array → returns (ROIs, ROIs) matrix.
    """
    fc_dict = OrderedDict()
    for sub in tqdm(ts_data, desc=desc):
        fc_dict[sub] = OrderedDict()
        for sess in ts_data[sub]:
            ts = ts_data[sub][sess]  # shape: (1200, 300)
            fc_dict[sub][sess] = compute_fn(ts)
    return fc_dict

### 1a. Precision Matrix (Ledoit-Wolf regularized inverse covariance)

The **precision matrix** $\Theta = \Sigma^{-1}$ captures direct (conditional) statistical
dependencies between regions. Ledoit-Wolf shrinkage regularisation ensures numerical
stability when $p$ (regions) is large relative to $n$ (timepoints).

In [ ]:
def compute_precision(ts):
    """Precision matrix via Ledoit-Wolf shrinkage.  ts: (timepoints, ROIs)."""
    lw = LedoitWolf()
    lw.fit(ts)
    return lw.precision_

print("Computing precision matrices (Ledoit-Wolf)...")
precision_dict = build_fc_dict(ts_data, compute_precision, desc="Precision")
print(f"Precision matrix shape: {precision_dict[subs[0]][0].shape}")

### 1b. Partial Correlation (derived from precision)

$$\rho_{ij|\text{rest}} = -\frac{\Theta_{ij}}{\sqrt{\Theta_{ii}\,\Theta_{jj}}}$$

In [ ]:
def precision_to_partial_corr(P):
    """Convert precision matrix to partial correlation matrix."""
    d = np.sqrt(np.diag(P))
    pcorr = -P / np.outer(d, d)
    np.fill_diagonal(pcorr, 1.0)
    return pcorr

print("Deriving partial correlation from precision matrices...")
pcorr_dict = OrderedDict()
for sub in tqdm(precision_dict, desc="Partial Corr"):
    pcorr_dict[sub] = OrderedDict()
    for sess in precision_dict[sub]:
        pcorr_dict[sub][sess] = precision_to_partial_corr(precision_dict[sub][sess])
print(f"Partial correlation matrix shape: {pcorr_dict[subs[0]][0].shape}")

### 1c. Gaussian Copula Mutual Information (GCMI)

GCMI (Ince et al., 2017) captures **nonlinear monotonic** dependencies:
1. Rank-transform each ROI time series
2. Map ranks to Gaussian quantiles (copula normalisation)
3. Compute MI assuming bivariate Gaussian copula:
   $\text{MI}(X,Y) = -\tfrac12 \log(1 - r_{\text{copula}}^2)$

In [ ]:
def copula_normalize(ts):
    """Copula-normalize each column: rank → Gaussian quantiles.  ts: (T, ROIs)."""
    n = ts.shape[0]
    ranks = np.apply_along_axis(rankdata, 0, ts)
    u = ranks / (n + 1)
    return norm.ppf(u)

def compute_gcmi_matrix(ts):
    """GCMI matrix.  ts: (timepoints, ROIs)."""
    ts_cop = copula_normalize(ts)
    r = np.corrcoef(ts_cop.T)
    r = np.clip(r, -0.9999, 0.9999)
    mi = -0.5 * np.log(1 - r**2)
    return mi

print("Computing GCMI (Gaussian Copula MI) matrices...")
gcmi_dict = build_fc_dict(ts_data, compute_gcmi_matrix, desc="GCMI")
print(f"GCMI matrix shape: {gcmi_dict[subs[0]][0].shape}")

## 2. Build Fingerprinting Similarity Matrices

For each FC method, flatten each subject's connectivity matrix and compute the
Pearson correlation between session-0 and session-1 vectors.
Result: $N \times N$ similarity matrix (same protocol as main paper).

In [ ]:
def build_sim_matrix(fc_dict):
    """Build inter-session fingerprinting similarity matrix.
    Returns (N × N) where (i,j) = corr(sub_j sess0, sub_i sess1).
    """
    list_s0 = [sub[0].flatten() for sub in fc_dict.values()]
    list_s1 = [sub[1].flatten() for sub in fc_dict.values()]
    n = len(list_s0)
    corr_mat = np.corrcoef(list_s0, list_s1)
    return corr_mat[n:, :n]

print("Building fingerprinting similarity matrices...")
sim_precision = build_sim_matrix(precision_dict)
sim_pcorr     = build_sim_matrix(pcorr_dict)
sim_gcmi      = build_sim_matrix(gcmi_dict)
print("Done — all similarity matrices computed.")

## 3. Fingerprinting Metrics

In [ ]:
methods = OrderedDict([
    ('Scaffold (freq)',   scaff_corr),
    ('Precision',         sim_precision),
    ('Partial Corr',      sim_pcorr),
    ('GCMI',              sim_gcmi),
    ('Pearson FC (full)', fc_corr_th100),
    ('Pearson FC (25%)',  fc_corr_th25),
    ('Pearson FC (1%)',   fc_corr_th1),
])

rows = []
for name, sim in methods.items():
    idiff, i_self, i_others = compute_idiff(sim, kind='corr')
    effect = compute_effect_size(sim, kind='corr')
    sr1, sr2 = compute_success_rate(sim, kind='corr')
    rows.append({
        'Method': name,
        'Idiff': round(idiff, 4),
        'I_self': round(i_self, 4),
        'I_others': round(i_others, 4),
        'Effect Size (d)': round(effect, 4),
        'SR s1→s2 (%)': sr1,
        'SR s2→s1 (%)': sr2,
        'SR mean (%)': round((sr1 + sr2) / 2, 1),
    })

df_metrics = pd.DataFrame(rows)
df_metrics = df_metrics.sort_values('Effect Size (d)', ascending=False).reset_index(drop=True)
display(df_metrics.style.highlight_max(
    subset=['Idiff', 'Effect Size (d)', 'SR mean (%)'], color='lightgreen'
))

## 4. Visualizations
### 4a. Similarity Matrix Heatmaps

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 16))
fig.suptitle(f'Fingerprinting Similarity Matrices — {COHORT_LABEL}',
             fontsize=18, y=1.02)

order = ['Scaffold (freq)', 'Precision', 'Partial Corr', 'GCMI',
         'Pearson FC (full)', 'Pearson FC (25%)', 'Pearson FC (1%)']
for idx, name in enumerate(order):
    ax = axes[idx // 4, idx % 4]
    plot_identifiability(methods[name], ax=ax, title=name, kind='corr')

axes[1, 3].axis('off')
plt.tight_layout()
plt.savefig(FIG_PATH + 'similarity_matrices_comparison.png', dpi=150, bbox_inches='tight')
plt.savefig(FIG_PATH + 'similarity_matrices_comparison.pdf', bbox_inches='tight')
plt.show()

### 4b. Bar Chart — Metric Comparison

In [ ]:
COLORS = {
    'Scaffold (freq)': '#e74c3c',
    'Precision':       '#3498db',
    'Partial Corr':    '#2ecc71',
    'GCMI':            '#9b59b6',
    'Pearson FC (full)': '#bdc3c7',
    'Pearson FC (25%)':  '#95a5a6',
    'Pearson FC (1%)':   '#7f8c8d',
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
ordered = df_metrics.sort_values('Effect Size (d)', ascending=True)

for ax, metric, label in zip(axes,
    ['Effect Size (d)', 'Idiff', 'SR mean (%)'],
    ["Cohen's d", 'Idiff', 'Success Rate (%)']):
    bars = ax.barh(ordered['Method'], ordered[metric],
                   color=[COLORS[m] for m in ordered['Method']])
    ax.set_xlabel(label, fontsize=14)
    ax.set_title(label, fontsize=16)
    ax.tick_params(axis='y', labelsize=12)
    xmax = ax.get_xlim()[1]
    for bar, val in zip(bars, ordered[metric]):
        fmt = f'{val:.2f}' if metric != 'SR mean (%)' else f'{val:.0f}%'
        ax.text(bar.get_width() + 0.01 * xmax,
                bar.get_y() + bar.get_height()/2, fmt, va='center', fontsize=11)

plt.tight_layout()
plt.savefig(FIG_PATH + 'metrics_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.savefig(FIG_PATH + 'metrics_comparison_bars.pdf', bbox_inches='tight')
plt.show()

### 4c. Violin Plots — Intra vs Inter-subject Similarity

In [ ]:
fig, axes = plt.subplots(1, len(methods), figsize=(4 * len(methods), 6), sharey=True)
fig.suptitle('Intra- vs Inter-subject Similarity', fontsize=16, y=1.03)

for idx, (name, sim) in enumerate(methods.items()):
    ax = axes[idx]
    intra = np.diag(sim)
    mask = ~np.eye(sim.shape[0], dtype=bool)
    inter = sim[mask]

    data = pd.DataFrame({
        'Similarity': np.concatenate([intra, inter]),
        'Type': ['Intra'] * len(intra) + ['Inter'] * len(inter)
    })

    sns.violinplot(data=data, x='Type', y='Similarity', ax=ax,
                   palette=['#e74c3c', '#3498db'], inner='box', cut=0)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('')
    if idx == 0:
        ax.set_ylabel('Pearson Similarity', fontsize=13)
    else:
        ax.set_ylabel('')

plt.tight_layout()
plt.savefig(FIG_PATH + 'violin_intra_inter.png', dpi=150, bbox_inches='tight')
plt.savefig(FIG_PATH + 'violin_intra_inter.pdf', bbox_inches='tight')
plt.show()

### 4d. Combined Effect Size & Success Rate (compact summary figure)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Effect size
for _, row in df_metrics.iterrows():
    ax1.scatter(row['Effect Size (d)'], row['Method'],
                color=COLORS[row['Method']], s=200, zorder=5, edgecolors='k')
ax1.axvline(x=0, color='grey', linestyle='--', alpha=0.5)
ax1.set_xlabel("Cohen's d (Effect Size)", fontsize=14)
ax1.set_title("Effect Size", fontsize=16)
ax1.grid(axis='x', alpha=0.3)

# Success rate
for _, row in df_metrics.iterrows():
    ax2.scatter(row['SR mean (%)'], row['Method'],
                color=COLORS[row['Method']], s=200, zorder=5, edgecolors='k')
ax2.set_xlabel('Success Rate (%)', fontsize=14)
ax2.set_title('Success Rate', fontsize=16)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_PATH + 'effect_size_vs_sr_dots.png', dpi=150, bbox_inches='tight')
plt.savefig(FIG_PATH + 'effect_size_vs_sr_dots.pdf', bbox_inches='tight')
plt.show()

## 5. Save Results

In [ ]:
results = {
    'sim_scaffold_freq': scaff_corr,
    'sim_fc_th100':      fc_corr_th100,
    'sim_fc_th25':       fc_corr_th25,
    'sim_fc_th1':        fc_corr_th1,
    'sim_precision':     sim_precision,
    'sim_pcorr':         sim_pcorr,
    'sim_gcmi':          sim_gcmi,
    'metrics':           df_metrics,
}

save_file = FIG_PATH + 'alternative_fc_comparison.pck'
with open(save_file, 'wb') as f:
    pk.dump(results, f)
print(f"Results saved to {save_file}")

# Save individual similarity matrices
for label, mat in [('precision', sim_precision), ('pcorr', sim_pcorr), ('gcmi', sim_gcmi)]:
    np.save(FIG_PATH + f'sim_{label}.npy', mat)
    print(f"  Saved sim_{label}.npy")

print("\nFinal summary:")
display(df_metrics[['Method', 'Effect Size (d)', 'SR mean (%)', 'Idiff']])

# Metrics table used for Table S5
df_metrics.to_csv(config.RESULTS + f"/alternative_fc_metrics_{COHORT}.csv", index=False)
print("Saved", config.RESULTS + f"/alternative_fc_metrics_{COHORT}.csv")


## Sensitivity of partial correlation to temporal filtering

Partial correlation (Ledoit-Wolf) on the 100 unrelated subjects, using the HCP 1200-release time series (300 cortical regions, z-scored) under each preprocessing variant. Reported in Sec. S5.

In [ ]:
from topofinger.hcp_IO import list_100unrelated

def _ts(variant, sub, k):
    g, b = variant.split("_", 1)
    x = np.load(f"{config.HCP_1200}/{variant}/ts/{sub}_rfMRI_REST{k}_LR_Schaefer300+Tian_{g}_{b}.npz")["timeseries"][:config.N_CORTICAL].T
    x = np.nan_to_num(x)
    return (x - x.mean(0)) / x.std(0)

def _pcorr(x):
    return precision_to_partial_corr(LedoitWolf().fit(x).precision_)

rows_sens = []
subs100 = sorted(list_100unrelated)
for variant in ["NoGSR_LowPassOnly", "NoGSR_Bandpass", "NoGSR_HighPassOnly", "NoGSR_NoFilt", "GSR_Bandpass"]:
    a = np.array([_pcorr(_ts(variant, s, 1)).ravel() for s in subs100])
    b = np.array([_pcorr(_ts(variant, s, 2)).ravel() for s in subs100])
    sim = np.corrcoef(a, b)[len(subs100):, :len(subs100)]
    sr1, sr2 = compute_success_rate(sim)
    rows_sens.append({"variant": variant, "SR mean (%)": (sr1 + sr2) / 2, "Effect Size (d)": round(compute_effect_size(sim), 2)})
    print(rows_sens[-1])
pd.DataFrame(rows_sens).to_csv(config.RESULTS + "/alternative_fc_lowpass_sensitivity.csv", index=False)
